<a href="https://colab.research.google.com/github/dinarojasocho-oss/LAB.-COLAB.02/blob/main/Copia_de_LAB_D2_FGD_PSAYAN_2026_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO CALIFICADO N° 02
## Fundamentos de Gestión de Datos

| | |
|:---|:---|
| **Curso** | Fundamentos de Gestión de Datos |
| **Semana** | 2 |
| **Tema** | Consultas básicas en SQLite: filtros, ordenamiento y resumen |
| **Herramienta** | Google Colab + SQLite |
| **Docente** | Pilar Rocío Sayán Mejía |
| **Semestre** | 2026-I |
| **Nombre** | **RAMIREZ GARCIA JAHOSHUA HIRAM** |
| **Fecha** | **23/05/2026** |

---

> **Indicación:** Ejecuta cada celda en orden y escribe tu respuesta en el espacio indicado antes de continuar.

## Caso contextual

El restaurante **"El Buen Sabor"** quiere analizar sus ventas del mes de marzo.
Necesita saber qué productos se venden más, en qué ciudades hay mayores ingresos
y cuánto dinero se ha generado en total.

Para eso construiremos una base de datos SQLite sencilla y practicaremos
consultas básicas de filtrado, ordenamiento y resumen.

---
## Preparación del entorno

El siguiente bloque crea la base de datos y carga los datos del restaurante.
**Ejecútalo una sola vez** antes de comenzar las actividades.

---

> **Nota sobre visualización de resultados**
>
> En SQLite con Python podemos ver datos usando `cursor.fetchall()`, pero ese método
> devuelve una **lista de tuplas** sin nombres de columna y sin respetar los alias
> definidos con `AS`. El resultado se ve así: `[(580.0,), (210.0,), ...]`, lo cual
> es difícil de leer.
>
> En este laboratorio usamos en su lugar la función **`mostrar_consulta()`**, que
> internamente llama a `pandas.read_sql_query()`. Esto muestra los resultados como
> una **tabla con encabezados claros** donde los alias (`AS`) aparecen correctamente
> como nombre de columna. Por eso verás siempre `mostrar_consulta(consulta)` en
> lugar de `cursor.fetchall()`.

### Nota para docente

La celda de preparación fue adaptada para demostraciones: cierra una conexión previa si existe,
elimina `restaurante.db` si ya fue creado y vuelve a construir la base desde cero.
Puedes ejecutarla tantas veces como necesites sin errores.
Además, las consultas se muestran con `pandas` para que los alias creados con `AS`
aparezcan como encabezados de columna.

In [ ]:
import sqlite3
import pandas as pd
import os
import gc

# ============================================================
# BLOQUE DE PREPARACIÓN: reiniciar la base de datos desde cero
# ============================================================
# Puedes ejecutar esta celda cuantas veces quieras.
# Cierra conexiones previas, elimina restaurante.db y lo recrea.

DB_NAME = "restaurante.db"

# 1) Cerrar cursor anterior si existe
try:
    cursor.close()
    print("Cursor anterior cerrado.")
except NameError:
    print("No había cursor anterior.")
except Exception:
    print("Cursor anterior ya estaba cerrado o no disponible.")

# 2) Cerrar conexión anterior si existe
try:
    conexion.close()
    print("Conexión anterior cerrada.")
except NameError:
    print("No había conexión anterior.")
except Exception:
    print("Conexión anterior ya estaba cerrada o no disponible.")

# 3) Liberar recursos pendientes
gc.collect()

# 4) Eliminar el archivo de base de datos si ya existe
if os.path.exists(DB_NAME):
    os.remove(DB_NAME)
    print(f"Base de datos anterior eliminada: {DB_NAME}")
else:
    print("No existía una base de datos anterior.")

# 5) Crear nueva conexión y nuevo cursor
conexion = sqlite3.connect(DB_NAME)
cursor = conexion.cursor()
print("Nueva conexión creada.")

# ============================================================
# Crear tablas
# ============================================================

cursor.execute("""
CREATE TABLE clientes (
    cliente_id   INTEGER PRIMARY KEY,
    nombre       TEXT,
    apellido     TEXT,
    ciudad       TEXT
);
""")

cursor.execute("""
CREATE TABLE productos (
    producto_id  INTEGER PRIMARY KEY,
    producto     TEXT,
    categoria    TEXT,
    precio       REAL
);
""")

cursor.execute("""
CREATE TABLE ventas (
    venta_id     INTEGER PRIMARY KEY,
    cliente_id   INTEGER,
    producto     TEXT,
    categoria    TEXT,
    ciudad       TEXT,
    monto        REAL,
    fecha_venta  TEXT
);
""")

# ============================================================
# Insertar datos
# ============================================================

cursor.executemany("INSERT INTO clientes VALUES (?, ?, ?, ?);", [
    (1, 'Ana',    'Torres',  'Lima'),
    (2, 'Luis',   'Quispe',  'Arequipa'),
    (3, 'Maria',  'Ramos',   'Lima'),
    (4, 'Carlos', 'Vega',    'Cusco'),
    (5, 'Sofia',  'Lopez',   'Trujillo'),
    (6, 'Diego',  'Mendoza', 'Lima'),
    (7, 'Lucia',  'Salas',   'Arequipa'),
    (8, 'Jorge',  'Castro',  'Piura'),
])

cursor.executemany("INSERT INTO productos VALUES (?, ?, ?, ?);", [
    (1, 'Pollo a la brasa',   'Comida',  58.00),
    (2, 'Parrilla familiar',  'Comida', 120.00),
    (3, 'Ceviche mixto',      'Comida',  75.00),
    (4, 'Limonada frozen',    'Bebida',  18.00),
    (5, 'Cafe americano',     'Bebida',  12.00),
    (6, 'Torta de chocolate', 'Postre',  22.00),
    (7, 'Helado artesanal',   'Postre',  16.00),
    (8, 'Combo ejecutivo',    'Menu',    35.00),
])

cursor.executemany("INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?);", [
    (1,  1, 'Pollo a la brasa',   'Comida', 'Lima',      580.00, '2026-03-01'),
    (2,  2, 'Combo ejecutivo',    'Menu',   'Arequipa',  210.00, '2026-03-01'),
    (3,  3, 'Parrilla familiar',  'Comida', 'Lima',      720.00, '2026-03-02'),
    (4,  4, 'Ceviche mixto',      'Comida', 'Cusco',     450.00, '2026-03-02'),
    (5,  5, 'Limonada frozen',    'Bebida', 'Trujillo',  180.00, '2026-03-03'),
    (6,  6, 'Parrilla familiar',  'Comida', 'Lima',      960.00, '2026-03-03'),
    (7,  7, 'Torta de chocolate', 'Postre', 'Arequipa',  220.00, '2026-03-04'),
    (8,  8, 'Pollo a la brasa',   'Comida', 'Piura',     640.00, '2026-03-04'),
    (9,  1, 'Cafe americano',     'Bebida', 'Lima',       96.00, '2026-03-05'),
    (10, 2, 'Helado artesanal',   'Postre', 'Arequipa',  128.00, '2026-03-05'),
    (11, 3, 'Combo ejecutivo',    'Menu',   'Lima',      350.00, '2026-03-06'),
    (12, 4, 'Parrilla familiar',  'Comida', 'Cusco',     840.00, '2026-03-06'),
    (13, 5, 'Ceviche mixto',      'Comida', 'Trujillo',  525.00, '2026-03-07'),
    (14, 6, 'Pollo a la brasa',   'Comida', 'Lima',      760.00, '2026-03-07'),
    (15, 7, 'Limonada frozen',    'Bebida', 'Arequipa',  144.00, '2026-03-08'),
    (16, 8, 'Combo ejecutivo',    'Menu',   'Piura',     280.00, '2026-03-08'),
    (17, 1, 'Parrilla familiar',  'Comida', 'Lima',     1080.00, '2026-03-09'),
    (18, 3, 'Torta de chocolate', 'Postre', 'Lima',      264.00, '2026-03-09'),
])

conexion.commit()

# ============================================================
# Función para mostrar consultas con encabezados y alias
# ============================================================
# IMPORTANTE: usamos pandas en lugar de fetchall() porque:
#   - fetchall() devuelve tuplas sin nombres de columna
#   - pandas muestra una tabla con encabezados y respeta los alias AS

def mostrar_consulta(consulta):
    return pd.read_sql_query(consulta, conexion)

print("Base de datos lista. Tablas: clientes, productos, ventas.")
print("Puedes volver a ejecutar esta celda cuando quieras reiniciar la demostración.")

Cursor anterior ya estaba cerrado o no disponible.
Conexión anterior cerrada.
Base de datos anterior eliminada: restaurante.db
Nueva conexión creada.
Base de datos lista. Tablas: clientes, productos, ventas.
Puedes volver a ejecutar esta celda cuando quieras reiniciar la demostración.


---
## Actividad 1 — Exploración inicial

Antes de filtrar datos, siempre conviene ver qué hay en la tabla.

### Consulta 1 — Ver todos los registros

`SELECT *` significa "trae todas las columnas".
`FROM ventas` indica la tabla de donde se leen los datos.
Esta consulta no aplica ningún filtro: devuelve **todas las filas**.

In [ ]:
# Consulta 1: ver todos los registros de ventas
consulta = """
SELECT *
FROM ventas;
"""

mostrar_consulta(consulta)

,venta_id,cliente_id,producto,categoria,ciudad,monto,fecha_venta
0,1,1,Pollo a la brasa,Comida,Lima,580.0,2026-03-01
1,2,2,Combo ejecutivo,Menu,Arequipa,210.0,2026-03-01
2,3,3,Parrilla familiar,Comida,Lima,720.0,2026-03-02
3,4,4,Ceviche mixto,Comida,Cusco,450.0,2026-03-02
4,5,5,Limonada frozen,Bebida,Trujillo,180.0,2026-03-03
5,6,6,Parrilla familiar,Comida,Lima,960.0,2026-03-03
6,7,7,Torta de chocolate,Postre,Arequipa,220.0,2026-03-04
7,8,8,Pollo a la brasa,Comida,Piura,640.0,2026-03-04
8,9,1,Cafe americano,Bebida,Lima,96.0,2026-03-05
9,10,2,Helado artesanal,Postre,Arequipa,128.0,2026-03-05


**Pregunta 1.1** ¿Cuántas filas aparecen en el resultado?

📝 **Tu respuesta:** *En total hay 18 filas*

---

**Pregunta 1.2** ¿Qué columna crees que es más útil para conocer el desempeño del restaurante? ¿Por qué?

📝 **Tu respuesta:** *Yo creo que el monto es el mas importante por que apartir de este podemos ver los ingresos por cada transaccion*

---
## Actividad 2 — Filtros con WHERE

`WHERE` permite seleccionar solo las filas que cumplen una condición.
`AND` agrega una segunda condición: ambas deben cumplirse al mismo tiempo.

### Consulta 2 — Filtrar ventas mayores a S/. 500

`WHERE monto > 500` indica que solo se incluyen filas donde el monto supera 500.
El símbolo `>` significa **mayor que**.

In [ ]:
# Consulta 2: ventas con monto mayor a 500
consulta = """
SELECT *
FROM ventas
WHERE monto > 500;
"""

mostrar_consulta(consulta)

,venta_id,cliente_id,producto,categoria,ciudad,monto,fecha_venta
0,1,1,Pollo a la brasa,Comida,Lima,580.0,2026-03-01
1,3,3,Parrilla familiar,Comida,Lima,720.0,2026-03-02
2,6,6,Parrilla familiar,Comida,Lima,960.0,2026-03-03
3,8,8,Pollo a la brasa,Comida,Piura,640.0,2026-03-04
4,12,4,Parrilla familiar,Comida,Cusco,840.0,2026-03-06
5,13,5,Ceviche mixto,Comida,Trujillo,525.0,2026-03-07
6,14,6,Pollo a la brasa,Comida,Lima,760.0,2026-03-07
7,17,1,Parrilla familiar,Comida,Lima,1080.0,2026-03-09


**Pregunta 2.1** ¿Cuántas ventas superan los S/. 500?

📝 **Tu respuesta:** *Segun la grafica se puede ver que hay 8 ventas que superan los 500*

---

**Pregunta 2.2** ¿Qué productos aparecen con mayor frecuencia en este resultado?

📝 **Tu respuesta:** *La Parrilla Familiar es el producto que aparece con mayor frecuencia*

### Consulta 3 — Filtrar con dos condiciones: WHERE + AND

`AND` agrega una segunda condición: **ambas** deben cumplirse al mismo tiempo.
Aquí filtramos ventas mayores a S/. 500 **y además** que sean de Lima.

In [ ]:
# Consulta 3: ventas mayores a 500 en Lima
consulta = """
SELECT *
FROM ventas
WHERE monto > 500
AND ciudad = 'Lima';
"""

mostrar_consulta(consulta)

,venta_id,cliente_id,producto,categoria,ciudad,monto,fecha_venta
0,1,1,Pollo a la brasa,Comida,Lima,580.0,2026-03-01
1,3,3,Parrilla familiar,Comida,Lima,720.0,2026-03-02
2,6,6,Parrilla familiar,Comida,Lima,960.0,2026-03-03
3,14,6,Pollo a la brasa,Comida,Lima,760.0,2026-03-07
4,17,1,Parrilla familiar,Comida,Lima,1080.0,2026-03-09


**Pregunta 3.1** ¿Cuántos resultados hay ahora comparado con la Consulta 2? ¿Por qué son menos?

📝 **Tu respuesta:** *Eso pasa por que en la consulta 3 se esta usando el operador AND lo que lo hacce mas restrictivo*

---

**Pregunta 3.2** ¿Qué pasa si cambias `'Lima'` por `'Arequipa'`? Pruébalo y describe el resultado.

📝 **Tu respuesta:** *Apareceria vacio ya que ninguno de ellos supera los 500*

---
## Actividad 2B — Filtros con OR

`OR` permite seleccionar filas que cumplen **al menos una** de las condiciones.
A diferencia de `AND` (que exige que **todas** se cumplan), con `OR` basta con que
una sola condición sea verdadera para que la fila aparezca en el resultado.

### Consulta 3B — Filtrar ventas de Lima O Arequipa

`OR` entre dos condiciones de ciudad recupera todas las filas que correspondan
a **cualquiera** de las dos ciudades.
Con `AND` en este caso no obtendríamos ningún resultado,
porque una venta no puede pertenecer a dos ciudades al mismo tiempo.

In [ ]:
# Consulta 3B: ventas de Lima o de Arequipa
consulta = """
SELECT *
FROM ventas
WHERE ciudad = 'Lima'
OR ciudad = 'Arequipa';
"""

mostrar_consulta(consulta)

,venta_id,cliente_id,producto,categoria,ciudad,monto,fecha_venta
0,1,1,Pollo a la brasa,Comida,Lima,580.0,2026-03-01
1,2,2,Combo ejecutivo,Menu,Arequipa,210.0,2026-03-01
2,3,3,Parrilla familiar,Comida,Lima,720.0,2026-03-02
3,6,6,Parrilla familiar,Comida,Lima,960.0,2026-03-03
4,7,7,Torta de chocolate,Postre,Arequipa,220.0,2026-03-04
5,9,1,Cafe americano,Bebida,Lima,96.0,2026-03-05
6,10,2,Helado artesanal,Postre,Arequipa,128.0,2026-03-05
7,11,3,Combo ejecutivo,Menu,Lima,350.0,2026-03-06
8,14,6,Pollo a la brasa,Comida,Lima,760.0,2026-03-07
9,15,7,Limonada frozen,Bebida,Arequipa,144.0,2026-03-08


**Pregunta 3B.1** ¿Cuántas ventas aparecen? ¿Por qué hay más filas que cuando usamos `AND`?

📝 **Tu respuesta:** *Aparecen 12 ventas. Esto es por al usar el operador OR basta con que se cummpla una de las condiciones para ser valido, lo cual hace que sea mas inclusivo y menos restricvtivo*

---

**Pregunta 3B.2** ¿Qué diferencia fundamental existe entre `AND` y `OR` al filtrar datos?

📝 **Tu respuesta:** *En AND se deben cumplir ambas condiciones para aser valido, en cambio en OR solo basta que se cumpla una de las conclusiones para ser valido.*

### Consulta 3C — Filtrar por categoría: Bebida O Postre

Aquí usamos `OR` para traer productos de dos categorías distintas en una sola consulta.
Si añadiéramos más categorías, basta con agregar más condiciones `OR`.

In [ ]:
# Consulta 3C: ventas de categoría Bebida o Postre
consulta = """
SELECT producto, categoria, ciudad, monto
FROM ventas
WHERE categoria = 'Bebida'
OR categoria = 'Postre';
"""

mostrar_consulta(consulta)

,producto,categoria,ciudad,monto
0,Limonada frozen,Bebida,Trujillo,180.0
1,Torta de chocolate,Postre,Arequipa,220.0
2,Cafe americano,Bebida,Lima,96.0
3,Helado artesanal,Postre,Arequipa,128.0
4,Limonada frozen,Bebida,Arequipa,144.0
5,Torta de chocolate,Postre,Lima,264.0


**Pregunta 3C.1** ¿Cuántos registros devuelve la consulta? ¿Qué categorías incluye?

📝 **Tu respuesta:** *Devuelve un total de 6 registros, las categorias son: Bebida y Postre.*

---

**Pregunta 3C.2** ¿Cómo modificarías la consulta para incluir también la categoría `'Menu'`?

📝 **Tu respuesta:** *Se modificaria de la siguiente forma: WHERE categoria = 'Bebida' OR categoria = 'Postre' OR categoria = 'Menu';*

---
## Actividad 3 — Ordenamiento

`ORDER BY` ordena los resultados según una columna.
`ASC` = de menor a mayor.  `DESC` = de mayor a menor.
`LIMIT` limita cuántas filas se muestran.

### Consulta 4 — Ordenar de menor a mayor (ASC)

`ORDER BY monto ASC` coloca primero las ventas más pequeñas.
Útil para identificar las ventas de menor valor.

In [ ]:
# Consulta 4: ordenar ventas por monto de menor a mayor
consulta = """
SELECT producto, ciudad, monto
FROM ventas
ORDER BY monto ASC;
"""

mostrar_consulta(consulta)

,producto,ciudad,monto
0,Cafe americano,Lima,96.0
1,Helado artesanal,Arequipa,128.0
2,Limonada frozen,Arequipa,144.0
3,Limonada frozen,Trujillo,180.0
4,Combo ejecutivo,Arequipa,210.0
5,Torta de chocolate,Arequipa,220.0
6,Torta de chocolate,Lima,264.0
7,Combo ejecutivo,Piura,280.0
8,Combo ejecutivo,Lima,350.0
9,Ceviche mixto,Cusco,450.0


**Pregunta 4.1** ¿Cuál es la venta de menor monto? ¿A qué producto corresponde?

📝 **Tu respuesta:** *La venta de menor monto es de 96.0 y es el producto: Cafe americano*

---

**Pregunta 4.2** ¿En qué ciudad se registraron las ventas más bajas?

📝 **Tu respuesta:** *En Lima*

### Consulta 5 — Ordenar de mayor a menor (DESC)

`ORDER BY monto DESC` coloca primero las ventas más altas.
Útil para ver qué productos generan más ingresos.

In [ ]:
# Consulta 5: ordenar ventas por monto de mayor a menor
consulta = """
SELECT producto, ciudad, monto
FROM ventas
ORDER BY monto DESC;
"""

mostrar_consulta(consulta)

,producto,ciudad,monto
0,Parrilla familiar,Lima,1080.0
1,Parrilla familiar,Lima,960.0
2,Parrilla familiar,Cusco,840.0
3,Pollo a la brasa,Lima,760.0
4,Parrilla familiar,Lima,720.0
5,Pollo a la brasa,Piura,640.0
6,Pollo a la brasa,Lima,580.0
7,Ceviche mixto,Trujillo,525.0
8,Ceviche mixto,Cusco,450.0
9,Combo ejecutivo,Lima,350.0


**Pregunta 5.1** ¿Cuál es la venta más alta? ¿En qué ciudad ocurrió?

📝 **Tu respuesta:** *La venta mas alta es de 1080 y ocurrio en la ciudad de Lima*

---

**Pregunta 5.2** ¿Qué diferencia notas entre el resultado ASC y DESC?

📝 **Tu respuesta:** *ASC organiza los datos de menor a mayor, en cambio DESC organiza los datos de mayor a menor.*

### Consulta 6 — Mostrar solo el Top 5 (LIMIT)

`LIMIT 5` muestra únicamente las primeras 5 filas del resultado.
Combinado con `ORDER BY DESC`, obtenemos las 5 ventas más altas.

In [ ]:
# Consulta 6: top 5 ventas más altas
consulta = """
SELECT producto, categoria, ciudad, monto
FROM ventas
ORDER BY monto DESC
LIMIT 5;
"""

mostrar_consulta(consulta)

,producto,categoria,ciudad,monto
0,Parrilla familiar,Comida,Lima,1080.0
1,Parrilla familiar,Comida,Lima,960.0
2,Parrilla familiar,Comida,Cusco,840.0
3,Pollo a la brasa,Comida,Lima,760.0
4,Parrilla familiar,Comida,Lima,720.0


**Pregunta 6.1** ¿Qué productos aparecen en el Top 5? ¿A qué categoría pertenecen?

📝 **Tu respuesta:** *En el top 5 aparecen Parrilla familiar y Pollo a la Brasa*

---

**Pregunta 6.2** ¿Por qué es útil `LIMIT` cuando una tabla tiene miles de filas?

📝 **Tu respuesta: *Para solo mostrar una cantidad definida de filas, es to es util cuando solo queremos ver una aparte de la tabla para entender su estructura*

---
## Actividad 4 — Resumen estadístico

SQL tiene funciones que calculan resúmenes de una columna numérica:
- `COUNT(*)` → cuenta cuántas filas hay
- `SUM(col)` → suma todos los valores
- `AVG(col)` → calcula el promedio
- `MAX(col)` → devuelve el valor más alto
- `MIN(col)` → devuelve el valor más bajo
- `DISTINCT` → elimina valores repetidos

### Consulta 7 — Valores únicos con DISTINCT

`DISTINCT` elimina los valores repetidos y muestra cada valor una sola vez.
Útil para saber qué categorías o ciudades existen en la tabla.

In [ ]:
# Consulta 7: categorías únicas en ventas
consulta = """
SELECT DISTINCT categoria AS categoria_unica
FROM ventas;
"""

mostrar_consulta(consulta)

,categoria_unica
0,Comida
1,Menu
2,Bebida
3,Postre


**Pregunta 7.1** ¿Cuántas categorías únicas existen?

📝 **Tu respuesta:** *De acuerdo a lo que se visualiza en la tabla, hay en total 4 categorias unicas*

---

**Pregunta 7.2** ¿Qué pasaría si quitaras la palabra `DISTINCT`? Pruébalo y describe la diferencia.

📝 **Tu respuesta:** *En este caso devolveria un total de 18 filas ya que mostraria cada una de las ventas registradas repitiendose muchas veces.*

### Consulta 8 — Contar registros con COUNT

`COUNT(*)` cuenta el total de filas que devuelve la consulta.
`AS total_ventas` le da un nombre al resultado.

In [ ]:
# Consulta 8: contar el total de ventas
consulta = """
SELECT COUNT(*) AS total_ventas
FROM ventas;
"""

mostrar_consulta(consulta)

,total_ventas
0,18


**Pregunta 8.1** ¿Cuántas ventas tiene registradas el restaurante?

📝 **Tu respuesta:** *En total hay 18 ventas registradas*

---

**Pregunta 8.2** ¿Cómo cambiarías la consulta para contar solo las ventas de Lima? *(Pista: añade WHERE)*

📝 **Tu respuesta:** **

### Consulta 9 — Sumar con SUM

`SUM(monto)` suma todos los valores de la columna `monto`.
Nos dice cuánto dinero ingresó en total.

In [ ]:
# Consulta 9: monto total vendido
consulta = """
SELECT SUM(monto) AS monto_total
FROM ventas;
"""

mostrar_consulta(consulta)

,monto_total
0,8427.0


**Pregunta 9.1** ¿Cuánto dinero ha generado el restaurante en total?

📝 **Tu respuesta:** *Ha generado en total 8427.0*

---

**Pregunta 9.2** ¿Cómo calcularías el total vendido solo en Lima? *(Pista: añade WHERE ciudad = 'Lima')*

📝 **Tu respuesta:** *Escribe aquí...*

### Consulta 10 — Promedio, máximo y mínimo (AVG, MAX, MIN)

Estas tres funciones dan un resumen estadístico rápido de los montos:
promedio, valor más alto y valor más bajo.

In [ ]:
# Consulta 10: promedio, máximo y mínimo de ventas
consulta = """
SELECT
    ROUND(AVG(monto), 2) AS promedio_monto,
    MAX(monto) AS monto_maximo,
    MIN(monto) AS monto_minimo
FROM ventas;
"""

mostrar_consulta(consulta)

,promedio_monto,monto_maximo,monto_minimo
0,468.17,1080.0,96.0


**Pregunta 10.1** ¿Cuál fue la venta más alta y la más baja?

📝 **Tu respuesta:** *La venta mas alta es de 1080.0 y la mas baja es de 96.0*

---

**Pregunta 10.2** El promedio de ventas es aproximadamente ¿cuánto? ¿Qué te indica ese número sobre el negocio?

📝 **Tu respuesta:** *El promedio es de 468.17. Este valor es uno que consideramos como el estandar*

---
## Conclusiones

Escribe tres conclusiones sobre lo que aprendiste en este laboratorio.

**1.** *Se aprendio a como usar WHERE, el uso de AND y de OR*

**2.** *Se aprendio a usar LIMIT y a usar ORDER BY para ver delimitar que valores vemos y por ejemplo identificar el top 5 por ejemplo*

**3.** *Se aprendio a usar SUM,COUNT, AVG, MAX y MIN para ver metricas de los registros*

In [ ]:
# Cerrar conexión explícitamente al terminar la clase o la demostración
try:
    cursor.close()
    print("Cursor cerrado correctamente.")
except Exception as e:
    print("No se pudo cerrar el cursor:", e)

try:
    conexion.close()
    print("Conexión cerrada correctamente.")
except Exception as e:
    print("No se pudo cerrar la conexión:", e)

print("Laboratorio completado.")

Cursor cerrado correctamente.
Conexión cerrada correctamente.
Laboratorio completado.


---
*2026-I — Fundamentos de Gestión de Datos — TECSUP — Docente: Pilar Rocío Sayán Mejía*